# SocialEngineerArena - Colab Training Notebook

This notebook is designed to run end-to-end in Google Colab.

It will:
1. Clone the repo
2. Install dependencies
3. Configure training vars
4. Run a dry-run sanity check
5. Run training
6. Show logs and artifacts (`loss_curve.png`, `summary.json`)


In [1]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vinod820/open_env_final.git"
REPO_DIR = Path("/content/social-engineer-arena")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo already exists:", REPO_DIR)

%cd /content/social-engineer-arena
!git pull


C:\content\social-engineer-arena
Already up to date.


In [2]:
# Colab dependency setup
!pip -q install -U pip
!pip -q install -U trl transformers datasets accelerate peft matplotlib torch huggingface_hub openenv-core
!pip -q install -e .

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


^C


ImportError: DLL load failed while importing _C: The specified module could not be found.

## (Optional) Hugging Face Login

Needed only if you want to push model artifacts to your HF model repo.


In [ ]:
from huggingface_hub import notebook_login
# Uncomment the next line if pushing to Hub:
# notebook_login()


In [ ]:
# Training configuration (safe defaults)
import os
from pathlib import Path

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

# Fast test settings (works quickly in Colab)
os.environ["MODEL_NAME"] = os.getenv("MODEL_NAME", "sshleifer/tiny-gpt2")
os.environ["OUTPUT_REPO"] = os.getenv("OUTPUT_REPO", "vinod2005/social-engineer-arena-suggest")
os.environ["OUTPUT_DIR"] = os.getenv("OUTPUT_DIR", "outputs/colab_training")
os.environ["SCENARIOS_PATH"] = os.getenv("SCENARIOS_PATH", "social_engineer_arena/data/scenarios.large.json")
os.environ["MAX_STEPS"] = os.getenv("MAX_STEPS", "10")
os.environ["DATA_MULTIPLIER"] = os.getenv("DATA_MULTIPLIER", "1")
os.environ["GRAD_ACCUM_STEPS"] = os.getenv("GRAD_ACCUM_STEPS", "1")
os.environ["EVAL_STRATEGY"] = os.getenv("EVAL_STRATEGY", "no")
os.environ["MAX_EVAL_SCENARIOS"] = os.getenv("MAX_EVAL_SCENARIOS", "1")
os.environ["GEN_MAX_NEW_TOKENS"] = os.getenv("GEN_MAX_NEW_TOKENS", "24")
os.environ["PUSH_TO_HUB"] = os.getenv("PUSH_TO_HUB", "0")

Path("outputs/logs").mkdir(parents=True, exist_ok=True)
print("Configured OUTPUT_DIR:", os.environ["OUTPUT_DIR"])
print("Configured MODEL_NAME:", os.environ["MODEL_NAME"])


In [ ]:
# Dry-run sanity check (quickly validates data + config)
import os
import subprocess

env = dict(os.environ)
env["DRY_RUN"] = "1"
res = subprocess.run(["python", "scripts/train_hf_job_sft.py"], env=env, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Dry-run failed. Fix this before full training.")


In [ ]:
# Full training run + log capture
import os
import subprocess
from datetime import datetime
from pathlib import Path

ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
log_path = Path(f"outputs/logs/colab_train_{ts}.log")
print("Log file:", log_path)

with log_path.open("w", encoding="utf-8") as f:
    proc = subprocess.Popen(
        ["python", "scripts/train_suggest_model.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=dict(os.environ),
    )
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    code = proc.wait()

if code != 0:
    raise RuntimeError(f"Training failed with exit code {code}. Check {log_path}")

print("Training completed successfully.")
print("Saved log:", log_path)


In [ ]:
# Show generated artifacts
from pathlib import Path
import json

out_dir = Path(os.environ["OUTPUT_DIR"])
print("Output directory:", out_dir)
print("Files:")
for p in sorted(out_dir.glob("*")):
    print(" -", p)

summary_path = out_dir / "summary.json"
if summary_path.exists():
    print("\nsummary.json:")
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))
else:
    print("No summary.json found")


In [ ]:
# Reward curve from baseline evaluation
!python scripts/evaluate_baselines.py
!python scripts/make_reward_plot.py

from pathlib import Path
reward_curve = Path("assets/reward_curve.png")
print("Reward curve exists:", reward_curve.exists(), reward_curve)


In [ ]:
# Optional: quick TRL GRPO run (RL framework evidence)
# Keep tiny settings so this stays lightweight in Colab.
import os
import subprocess

os.environ["MODEL_NAME"] = os.getenv("MODEL_NAME", "sshleifer/tiny-gpt2")
os.environ["OUTPUT_DIR"] = os.getenv("OUTPUT_DIR", "outputs/grpo_colab")
os.environ["MAX_STEPS"] = os.getenv("MAX_STEPS", "4")
os.environ["NUM_GENERATIONS"] = os.getenv("NUM_GENERATIONS", "2")
os.environ["MAX_PROMPTS"] = os.getenv("MAX_PROMPTS", "64")

res = subprocess.run(["python", "scripts/train_trl_grpo.py"], text=True)
if res.returncode != 0:
    raise RuntimeError("GRPO run failed. Check prior cells and package versions.")
print("GRPO run complete.")


## For better-quality training

After the quick successful run, increase:

- `MODEL_NAME` -> `Qwen/Qwen2.5-0.5B-Instruct`
- `MAX_STEPS` -> `120` or higher
- `DATA_MULTIPLIER` -> `8` to `32`
- `PUSH_TO_HUB` -> `1` (and login first)

Then re-run the training cell.
